# DATA INTENSIVE SYSTEMS IN BIOMEDICINE

## Opis projektu: ....

Tu się kiedyś coś dopisze

---
## Wymogi: 
* ``Przeprowadź czyszczenie danych (jesli jest to wymagane lub nie – w obu przypadkach uzasadnij), zweryfikuj ilość pustych wartości i czy mają jakiś wpływ na analizę ``
* ``Przedstaw klasyfikację danych ``
* ``Wykorzystaj MLlib do predykcji ``
* ``Uwzględnij wizualizację danych za pomocą bibliotek pandas lub matplotlib.``
* ``Skorzystaj z paradygmatu MapReduce oraz funkcji lambda. ``
---

## Wczytanie danych za pomocą spark i wyświetlenie 10 pierwszych wierszy

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Diabetes Analysis") \
    .getOrCreate()

DATA_PATH = "/app/data/diabetes_health_indicators.csv"

df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

df.show(10, truncate=False)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 19:18:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|Diabetes_012|HighBP|HighChol|CholCheck|BMI |Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex|Age |Education|Income|
+------------+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|0.0         |1.0   |1.0     |1.0      |40.0|1.0   |0.0   |0.0                 |0.0         |0.0   |1.0    |0.0              |1.0          |0.0        |5.0    |18.0    |15.0    |1.0     |0.0|9.0 |4.0      |3.0   |
|0.0         |0.0   |0.0     |0.0      |25.0|1.0   |0.0   |0.0                 |1.0         |0.0   |0.0    |0.0              |0.0          |1.0 

In [27]:
print("-" * 15 + "Informacje o Strukturze bazy danych" + "-" * 15)
print(f'Całkowita ilość wierszy: {df.count()}')
print(f'Całkowita ilość kolumn: {len(df.columns)}')
print(f'Nazwy kolumn: {df.columns}')
print(65 * "-")

---------------Informacje o Strukturze bazy danych---------------


Całkowita ilość wierszy: 253680
Całkowita ilość kolumn: 22
Nazwy kolumn: ['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
-----------------------------------------------------------------


## Sprawdzenie czy występują puste wiersze

In [ ]:
from pyspark.sql.functions import col, count, when

# Zliczanie brakujących (null) wartości w każdej kolumnie
missing_values = df.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df.columns
])

missing_values.show(truncate=False)

+------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+---+---------+------+
|Diabetes_012|HighBP|HighChol|CholCheck|BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex|Age|Education|Income|
+------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+---+---------+------+
|0           |0     |0       |0        |0  |0     |0     |0                   |0           |0     |0      |0                |0            |0          |0      |0       |0       |0       |0  |0  |0        |0     |
+------------+------+--------+---------+---+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-

Dla każdej kolumny sprawdzono liczbę wartości NULL. Wyniki pokazują, że:

- żadna kolumna nie zawiera brakujących danych,
- wszystkie wartości wynoszą 0 (brak braków).

W związku z tym zbiór danych nie wymagał procesu czyszczenia. Brak brakujących danych oznacza, że dane są kompletne i mogą zostać bezpośrednio wykorzystane do dalszej analizy oraz budowy modelu ML.

---

## Przedstawienie klasyfikacji danych

In [28]:
df.printSchema()

root
 |-- Diabetes_012: double (nullable = true)
 |-- HighBP: double (nullable = true)
 |-- HighChol: double (nullable = true)
 |-- CholCheck: double (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: double (nullable = true)
 |-- Stroke: double (nullable = true)
 |-- HeartDiseaseorAttack: double (nullable = true)
 |-- PhysActivity: double (nullable = true)
 |-- Fruits: double (nullable = true)
 |-- Veggies: double (nullable = true)
 |-- HvyAlcoholConsump: double (nullable = true)
 |-- AnyHealthcare: double (nullable = true)
 |-- NoDocbcCost: double (nullable = true)
 |-- GenHlth: double (nullable = true)
 |-- MentHlth: double (nullable = true)
 |-- PhysHlth: double (nullable = true)
 |-- DiffWalk: double (nullable = true)
 |-- Sex: double (nullable = true)
 |-- Age: double (nullable = true)
 |-- Education: double (nullable = true)
 |-- Income: double (nullable = true)



### Opis struktury danych

Dzięki zastosowaniu funkcji `printSchema()` możliwe jest wyświetlenie struktury analizowanej bazy danych. Analizując wynik funkcji można zauważyć, że:
- wszystkie kolumny w zbiorze danych są typu numerycznego (`double`),
- każda kolumna dopuszcza wartości NULL (nullable = true), jednak wcześniejsza analiza wykazała brak brakujących danych.

In [29]:
df.describe().show(truncate=False)

+-------+-------------------+------------------+-------------------+------------------+------------------+-------------------+-------------------+--------------------+------------------+------------------+-------------------+-------------------+-------------------+-------------------+------------------+------------------+------------------+-------------------+-------------------+-----------------+------------------+------------------+
|summary|Diabetes_012       |HighBP            |HighChol           |CholCheck         |BMI               |Smoker             |Stroke             |HeartDiseaseorAttack|PhysActivity      |Fruits            |Veggies            |HvyAlcoholConsump  |AnyHealthcare      |NoDocbcCost        |GenHlth           |MentHlth          |PhysHlth          |DiffWalk           |Sex                |Age              |Education         |Income            |
+-------+-------------------+------------------+-------------------+------------------+------------------+----------------

### Podział zmiennych

Na podstawie analizy zbioru danych można wyróżnić kilka typów zmiennych, które się w nim znajdują:


### 1. Zmienna docelowa


- **Diabetes_012** – status cukrzycy:
  - 0 – brak cukrzycy
  - 1 – stan przedcukrzycowy
  - 2 – cukrzyca



### 2. Zmienne binarne


- **HighBP** – nadciśnienie  
  - 0 – brak  
  - 1 – występuje  

- **HighChol** – wysoki cholesterol  
  - 0 – brak  
  - 1 – występuje  

- **CholCheck** – badanie cholesterolu w ciągu ostatnich 5 lat  
  - 0 – brak  
  - 1 – tak  

- **Smoker** – czy kiedykolwiek wypalono ≥100 papierosów  
  - 0 – nie  
  - 1 – tak  

- **Stroke** – przebyty udar  
  - 0 – nie  
  - 1 – tak  

- **HeartDiseaseorAttack** – choroba serca lub zawał  
  - 0 – nie  
  - 1 – tak  

- **PhysActivity** – aktywność fizyczna w ostatnich 30 dniach  
  - 0 – brak  
  - 1 – tak  

- **Fruits** – spożycie owoców ≥1 raz dziennie  
  - 0 – nie  
  - 1 – tak  

- **Veggies** – spożycie warzyw ≥1 raz dziennie  
  - 0 – nie  
  - 1 – tak  

- **HvyAlcoholConsump** – nadmierne spożycie alkoholu  
  - 0 – nie  
  - 1 – tak  

- **AnyHealthcare** – dostęp do opieki zdrowotnej  
  - 0 – brak  
  - 1 – tak  

- **NoDocbcCost** – brak wizyty lekarskiej z powodu kosztów  
  - 0 – nie  
  - 1 – tak  

- **DiffWalk** – trudności w chodzeniu lub wchodzeniu po schodach  
  - 0 – brak  
  - 1 – tak  

- **Sex**  
  - 0 – female  
  - 1 – male  


### 3. Zmienne ciągłe

- **BMI**  
  - ciągła zmienna numeryczna  
  - zakres: 12 – 98  



### 4. Zmienne kategoryczne


- **GenHlth** – ogólny stan zdrowia (1–5)  
  - 1 – excellent  
  - 2 – very good  
  - 3 – good  
  - 4 – fair  
  - 5 – poor  

- **MentHlth** – liczba dni gorszego samopoczucia psychicznego (0–30)

- **Age** – kategoria wieku (1–13) 

- **PhysHlth** – liczba dni niezdrowego trybu życia (0–30)

- **Education** – poziom edukacji (1–6)  
  - 1 – brak edukacji / przedszkole  
  - 6 – wyższe  

- **Income** – poziom dochodu (1–8)  
  - 1 – < $10,000  
  - 8 – ≥ $75,000  

Dla zmiennych **binarnych i kategorycznych** zastosowano analizę rozkładu wartości. Żeby zapoznać się z rozkładem tego typu danych.

In [35]:
categorical_columns = [c for c in df.columns if c != "BMI"]

total_count = df.count()

for col_name in categorical_columns:
    print(f"\nKolumna: {col_name}")
    
    df.groupBy(col_name) \
      .count() \
      .orderBy(col_name) \
      .show(truncate=False)


Kolumna: Diabetes_012


+------------+------+
|Diabetes_012|count |
+------------+------+
|0.0         |213703|
|1.0         |4631  |
|2.0         |35346 |
+------------+------+


Kolumna: HighBP


+------+------+
|HighBP|count |
+------+------+
|0.0   |144851|
|1.0   |108829|
+------+------+


Kolumna: HighChol


+--------+------+
|HighChol|count |
+--------+------+
|0.0     |146089|
|1.0     |107591|
+--------+------+


Kolumna: CholCheck


+---------+------+
|CholCheck|count |
+---------+------+
|0.0      |9470  |
|1.0      |244210|
+---------+------+


Kolumna: Smoker


+------+------+
|Smoker|count |
+------+------+
|0.0   |141257|
|1.0   |112423|
+------+------+


Kolumna: Stroke


... (5 outputs left out)

+--------+------+
|MentHlth|count |
+--------+------+
|0.0     |175680|
|1.0     |8538  |
|2.0     |13054 |
|3.0     |7381  |
|4.0     |3789  |
|5.0     |9030  |
|6.0     |988   |
|7.0     |3100  |
|8.0     |639   |
|9.0     |91    |
|10.0    |6373  |
|11.0    |41    |
|12.0    |398   |
|13.0    |41    |
|14.0    |1167  |
|15.0    |5505  |
|16.0    |88    |
|17.0    |54    |
|18.0    |97    |
|19.0    |16    |
+--------+------+
only showing top 20 rows


Kolumna: PhysHlth


+--------+------+
|PhysHlth|count |
+--------+------+
|0.0     |160052|
|1.0     |11388 |
|2.0     |14764 |
|3.0     |8495  |
|4.0     |4542  |
|5.0     |7622  |
|6.0     |1330  |
|7.0     |4538  |
|8.0     |809   |
|9.0     |179   |
|10.0    |5595  |
|11.0    |60    |
|12.0    |578   |
|13.0    |68    |
|14.0    |2587  |
|15.0    |4916  |
|16.0    |112   |
|17.0    |96    |
|18.0    |152   |
|19.0    |22    |
+--------+------+
only showing top 20 rows


Kolumna: DiffWalk


+--------+------+
|DiffWalk|count |
+--------+------+
|0.0     |211005|
|1.0     |42675 |
+--------+------+


Kolumna: Sex


+---+------+
|Sex|count |
+---+------+
|0.0|141974|
|1.0|111706|
+---+------+


Kolumna: Age
+----+-----+
|Age |count|
+----+-----+
|1.0 |5700 |
|2.0 |7598 |
|3.0 |11123|
|4.0 |13823|
|5.0 |16157|
|6.0 |19819|
|7.0 |26314|
|8.0 |30832|
|9.0 |33244|
|10.0|32194|
|11.0|23533|
|12.0|15980|
|13.0|17363|
+----+-----+


Kolumna: Education
+---------+------+
|Education|count |
+---------+------+
|1.0      |174   |
|2.0      |4043  |
|3.0      |9478  |
|4.0      |62750 |
|5.0      |69910 |
|6.0      |107325|
+---------+------+


Kolumna: Income
+------+-----+
|Income|count|
+------+-----+
|1.0   |9811 |
|2.0   |11783|
|3.0   |15994|
|4.0   |20135|
|5.0   |25883|
|6.0   |36470|
|7.0   |43219|
|8.0   |90385|
+------+-----+



## Mapreduce